In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.nn.utils.rnn import pad_sequence
torch.manual_seed(1232)
import random

In [91]:
a = torch.randint(0, 20, (10, ))
b = torch.randint(0, 20, (10, ))
c = a + b
print("a=>", a)
print("b=>", b)
print("c=>", c)

a=> tensor([ 6, 19, 13,  0, 18, 16, 13, 14, 15,  7])
b=> tensor([16,  2,  4,  7,  2, 16, 16,  5,  2,  2])
c=> tensor([22, 21, 17,  7, 20, 32, 29, 19, 17,  9])


In [92]:
chars = "1234567890+=."
vocab = sorted(set(chars))
vocab_size = len(vocab)
stoi = {ch:i for i, ch in enumerate(vocab)}
itos = {i:ch for i, ch in enumerate(vocab)}
encode = lambda s: [stoi[ch] for ch in s]
decode = lambda l: ''.join([itos[i] for i in l])

In [93]:
print(encode("1+2="))
decode(encode("1+2="))

[3, 0, 4, 12]


'1+2='

In [130]:
def data_generator():
    a = random.randint(0, 20)
    b = random.randint(0, 20)
    X = f"{a}+{b}={str(a+b)[::-1]}"
    maskLen = len(f"{a}+{b}=")
    Y = [-1] * maskLen + encode(str(a+b)[::-1])
    return X, Y

In [134]:
BATCH_SIZE = 32
def get_batch():
    X, Y = [], []
    batch = map(lambda _: data_generator(), range(BATCH_SIZE))
    for b in batch:
        X.append(torch.tensor(encode(b[0])))
        Y.append(torch.tensor(b[1]))
    
    X = pad_sequence(X, padding_value=encode(".")[0], batch_first=True, padding_side='left')
    Y = pad_sequence(Y, padding_value=-1, batch_first=True, padding_side='left')

    return X, Y

In [124]:
a, b = data_generator()
a, b

('15+9=42', [-1, -1, -1, -1, -1, 6, 4])

In [136]:
X, Y = get_batch()

In [137]:
X.shape, Y.shape

(torch.Size([32, 8]), torch.Size([32, 8]))

In [115]:
a, b = data_generator()
print(a)
print(b)

14+15=
14+15=92


In [104]:
a = 5
b = 12
print(f"{a}+{b}={str(a+b)[::-1]}")

5+12=71


In [125]:
stoi

{'+': 0,
 '.': 1,
 '0': 2,
 '1': 3,
 '2': 4,
 '3': 5,
 '4': 6,
 '5': 7,
 '6': 8,
 '7': 9,
 '8': 10,
 '9': 11,
 '=': 12}

In [ ]:
a = torch.randint(0,5,(9,5)).float()
out = [1, 1, 1, 1, 1, 1, 2, 3, 4]
out = torch.tensor(out)
ce = F.cross_entropy(a, out)
ce

tensor(2.8649)

In [152]:
a.shape, out.shape

(torch.Size([9, 2]), torch.Size([9]))

In [161]:
ignore_index = 1
exp_a = torch.exp(a)
probs = exp_a / exp_a.sum(dim=1, keepdim=True)
correct_probs = probs[torch.arange(len(out)), out]
log_probs = -torch.log(correct_probs)
mask = (out != ignore_index)
loss = log_probs[mask].mean()
print(loss)

tensor(2.8649)


In [159]:
ce = F.cross_entropy(a, out)
ce

tensor(2.4743)

In [160]:
probs.shape

torch.Size([9, 5])

In [162]:
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.nn.utils.rnn import pad_sequence
import random
torch.manual_seed(1232)


chars = "1234567890+=."
vocab = sorted(set(chars))
vocab_size = len(vocab)
stoi = {ch:i for i, ch in enumerate(vocab)}
itos = {i:ch for i, ch in enumerate(vocab)}
encode = lambda s: [stoi[ch] for ch in s]
decode = lambda l: ''.join([itos[i] for i in l])

# =============== HYPERPARAMETERS ===============
BATCH_SIZE = 32
BLOCK_SIZE = 8
MAX_ITERS = 2000
EVAL_INTERVAL = 300
LR = 1e-3
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
EVAL_ITERS = 200
N_EMBD = 32

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(EVAL_ITERS)
        for k in range(EVAL_ITERS):
            X, Y = get_batch(BATCH_SIZE)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

def data_generator():
    a = random.randint(0, 20)
    b = random.randint(0, 20)
    X = f"{a}+{b}={str(a+b)[::-1]}"
    maskLen = len(f"{a}+{b}=")
    Y = [-1] * maskLen + encode(str(a+b)[::-1])
    return X, Y

def get_batch(size):
    X, Y = [], []
    batch = map(lambda _: data_generator(), range(size))
    for b in batch:
        X.append(torch.tensor(encode(b[0])))
        Y.append(torch.tensor(b[1]))
    
    X = pad_sequence(X, padding_value=encode(".")[0], batch_first=True, padding_side='left')
    Y = pad_sequence(Y, padding_value=-1, batch_first=True, padding_side='left')

    return X, Y


class CausalSelfAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        # Created a new variable called c_attn (causal attention). As input is N_EMBD, output size would 3 * head_size for K, Q, V
        self.c_attn = nn.Linear(N_EMBD, 3 * N_EMBD, bias=False)
        self.proj = nn.Linear(N_EMBD, N_EMBD) # projection layer that goes back in the pathway
        self.head_size = head_size
        self.num_heads = num_heads
        self.register_buffer('tril', torch.tril(torch.ones(BLOCK_SIZE, BLOCK_SIZE)).view(1, 1, BLOCK_SIZE, BLOCK_SIZE))
    
    def forward(self, x):
        B, T, C = x.shape
        x = self.c_attn(x) # applied a linear layer to fit the shape we desire as per head_size
        q, k, v = x.split(N_EMBD, dim=2) # since, we want to split on the channel dimension
        q = q.view(B, T, self.num_heads, self.head_size).transpose(1, 2) # (B, T, nh, head_size) => (B, nh, T, head_size)
        k = k.view(B, T, self.num_heads, self.head_size).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_size).transpose(1, 2)
        
        att = q @ k.transpose(-2, -1) * (self.head_size ** (-0.5))
        att = att.masked_fill(self.tril[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        att = att @ v # (B, nh, T, T) @ (B, nh, T, head_size) => (B, nh, T, head_size)
        att = att.transpose(1, 2).contiguous().view(B, T, C) # we have to use contiguous() to make the memory in tensors so that view() can work properly
        out = self.proj(att)
        return out

class FeedForward(nn.Module):
    """a simple linear layer followed by a non-linearity or a basic MLP block"""
    def __init__(self, N_EMBD):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(N_EMBD, 4 * N_EMBD),
            nn.ReLU(),
            nn.Linear(4 * N_EMBD, N_EMBD), # projection layer
        )
    
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """Transformers block: communication (done by self-attention) followed by computation (done by feedforward layer)"""
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = CausalSelfAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)  # normalizes at token level
        self.ln2 = nn.LayerNorm(n_embd)
    
    def forward(self, x):
        x = x + self.sa(self.ln1(x)) # added skip connection here
        x = x + self.ffwd(self.ln2(x)) # added skip connection here
        return x

class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, N_EMBD) # we are adding an intermediate layer instead of directly taking logits from the embedding table
        self.position_embedding_table = nn.Embedding(BLOCK_SIZE, N_EMBD) # so each block or each time component get its own positional embedding
        self.blocks = nn.Sequential(
            Block(n_embd=N_EMBD, n_head=4),
            Block(n_embd=N_EMBD, n_head=4),
            Block(n_embd=N_EMBD, n_head=4),
            nn.LayerNorm(N_EMBD),
        )
        self.lm_head = nn.Linear(N_EMBD, vocab_size) # adding a linear layer
        
    
    def forward(self, idx, targets=None):
        # idx and targets are both (B, T) tensor of integers
        B, T = idx.shape
        tok_embd = self.token_embedding_table(idx) # (B, T, C)
        pos_embd = self.position_embedding_table(torch.arange(T, device=DEVICE)) # T, C. So, for each index, I am getting an embedding that has the information of the position
        x = tok_embd + pos_embd # now x has both the information of identity and position. Although not much useful for bigram but conceptually important
        x = self.blocks(x)
        logits = self.lm_head(x) # (B, T, vocab_size)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets, ignore_index=-1) # we cannot call like this. This expects Channel (or C) dimension before. So, we have to reshape the logits
        
        return logits, loss
    
    
    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        # get the predictions
        for _ in range(max_new_tokens):
            # crop idx to the last bock_size tokens
            idx_cond = idx[:, -BLOCK_SIZE:]
            logits, loss = self(idx_cond)
            # focus only on the last time step (cuz bigram model)
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1)  # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        
        return idx
    
model = GPT()
m = model.to(DEVICE)
optimizer = torch.optim.AdamW(m.parameters(), lr=LR)

for iter in range(MAX_ITERS):
    
    if iter % EVAL_INTERVAL == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
    
    # batch from sample
    xb, yb = get_batch(BATCH_SIZE)
    
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    
    loss.backward()
    
    optimizer.step()
        

step 0: train loss 2.6190, val loss 2.6107
step 300: train loss 0.0154, val loss 0.0154
step 600: train loss 0.0047, val loss 0.0047
step 900: train loss 0.0023, val loss 0.0023
step 1200: train loss 0.0014, val loss 0.0013
step 1500: train loss 0.0009, val loss 0.0009
step 1800: train loss 0.0006, val loss 0.0006


In [170]:
correct = 0
for _ in range(50):
    a = random.randint(0, 20)
    b = random.randint(0, 20)
    prompt = f"{a}+{b}="
    print(prompt)
    context = torch.tensor(encode(prompt), dtype=torch.long).unsqueeze(0)
    output = m.generate(context, max_new_tokens=2)[0].tolist()
    print(decode(output))
    predicted = decode(output[len(prompt):])[::-1]
    print(decode(list(context)), predicted)
    if predicted == str(a+b):
        correct += 1

print(f"Accuracy: {correct}/50")

8+14=
8+14=77


KeyError: tensor([10,  0,  3,  6, 12])